In [ ]:
# TERMINAL COMMAND
'''
cd "/Users/matt.fritz/Desktop/Research Insights/Essay Prototype"
caffeinate -di python strategic_sweep_v1.0.py
'''

In [15]:
# DETERMINE WHICH SPLITS FAILED SWEEP

from pathlib import Path
import os
import pandas as pd
import yaml

# ── CONFIG ────────────────────────────────────────────────────────────────────
ROOT = Path(r"/Users/matt.fritz/Desktop/Research Insights/Essay Prototype")

SPLITS_YAML = ROOT / "CONFIG" / "splits.yaml"
OUTPUT_ROOT = ROOT / "OUTPUTS" / "runs" / "strategic_area"
CURATED_FILENAME = "curated_df.csv"
# ─────────────────────────────────────────────────────────────────────────────


def enumerate_expected_combos(splits_yaml: Path) -> pd.DataFrame:
    """Return expected (strategic_area_id, split_id) combos from splits.yaml."""
    with open(splits_yaml, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    run_plan = data.get("run_plan", {}) or {}
    rows = []
    skipped = []

    for area, spec in run_plan.items():
        spec = spec or {}

        first = spec.get("first_split")
        if isinstance(first, str):
            rows.append({
                "strategic_area_id": area,
                "split_id": first,
                "split_source": "first_split",
            })
        elif first is not None:
            skipped.append({
                "strategic_area_id": area,
                "split_source": "first_split",
                "detail": repr(first),
            })

        for item in spec.get("second_splits") or []:
            if isinstance(item, str):
                rows.append({
                    "strategic_area_id": area,
                    "split_id": item,
                    "split_source": "second_splits",
                })
            else:
                skipped.append({
                    "strategic_area_id": area,
                    "split_source": "second_splits",
                    "detail": repr(item),
                })

    if skipped:
        print("WARNING: skipped non-string split entries:")
        print(pd.DataFrame(skipped).to_string(index=False))
        print()

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["strategic_area_id", "split_id"])
        .sort_values(["strategic_area_id", "split_id"])
        .reset_index(drop=True)
    )


def find_curated_files(output_root: Path) -> list[Path]:
    """Find all curated_df.csv files under OUTPUTS/runs/strategic_area."""
    matches = []
    for dirpath, _, filenames in os.walk(output_root):
        for fname in filenames:
            if fname.lower() == CURATED_FILENAME.lower():
                matches.append(Path(dirpath) / fname)
    return sorted(matches)


def read_completed_from_curated(path: Path, output_root: Path) -> dict:
    """Read one curated_df.csv and infer its area/split."""
    df = pd.read_csv(path, nrows=50)

    area = None
    split = None

    if "strategic_area_id" in df.columns and df["strategic_area_id"].notna().any():
        area = str(df["strategic_area_id"].dropna().iloc[0]).strip()

    if "split_id" in df.columns and df["split_id"].notna().any():
        split = str(df["split_id"].dropna().iloc[0]).strip()

    rel_parts = path.relative_to(output_root).parts
    if area is None and len(rel_parts) >= 1:
        area = rel_parts[0]

    if split is None:
        if (
            "resolved_groupby_field" in df.columns
            and df["resolved_groupby_field"].notna().any()
        ):
            split = str(df["resolved_groupby_field"].dropna().iloc[0]).strip()

    return {
        "strategic_area_id": area,
        "split_id": split,
        "curated_path": str(path),
        "curated_mtime": path.stat().st_mtime,
    }


def load_completed_combos(output_root: Path) -> pd.DataFrame:
    """Return completed combos based on curated_df.csv files."""
    rows = []

    for path in find_curated_files(output_root):
        try:
            rows.append(read_completed_from_curated(path, output_root))
        except Exception as e:
            print(f"WARNING: could not read {path}: {type(e).__name__}: {e}")

    if not rows:
        return pd.DataFrame(columns=["strategic_area_id", "split_id"])

    return (
        pd.DataFrame(rows)
        .sort_values("curated_mtime", ascending=False)
        .drop_duplicates(["strategic_area_id", "split_id"], keep="first")
        .sort_values(["strategic_area_id", "split_id"])
        .reset_index(drop=True)
    )


def print_missing_yaml(missing_df: pd.DataFrame) -> None:
    """Print missing combos as a paste-ready splits.yaml run_plan block."""
    print("\nPaste-ready YAML for missing combos:")
    print("────────────────────────────────────")
    print("run_plan:")

    if missing_df.empty:
        print("  # No missing combos found.")
        return

    for area, g in (
        missing_df
        .sort_values(["strategic_area_id", "split_id"])
        .groupby("strategic_area_id", sort=True)
    ):
        splits = [str(x) for x in g["split_id"].dropna().tolist()]

        print(f"  {area}:")

        if "strategic_injected_tag" in splits:
            print("    first_split: strategic_injected_tag")

        second_splits = [s for s in splits if s != "strategic_injected_tag"]
        if second_splits:
            print("    second_splits:")
            for split in second_splits:
                print(f"      - {split}")


expected = enumerate_expected_combos(SPLITS_YAML)
completed = load_completed_combos(OUTPUT_ROOT)

expected_keys = expected[["strategic_area_id", "split_id"]].drop_duplicates()
completed_keys = completed[["strategic_area_id", "split_id"]].drop_duplicates()

status = expected_keys.merge(
    completed_keys.assign(has_curated_df=True),
    on=["strategic_area_id", "split_id"],
    how="left",
)
status["has_curated_df"] = status["has_curated_df"].fillna(False)

missing = status[~status["has_curated_df"]].copy()

print("Sweep combo check")
print("─────────────────")
print(f"Expected combos      : {len(expected_keys):,}")
print(f"Completed curated_df : {len(completed_keys):,}")
print(f"Missing curated_df   : {len(missing):,}")

if not missing.empty:
    print("\nMissing combos:")
    for _, row in missing.sort_values(["strategic_area_id", "split_id"]).iterrows():
        print(f"  - {row['strategic_area_id']} × {row['split_id']}")

print_missing_yaml(missing)

Sweep combo check
─────────────────
Expected combos      : 46
Completed curated_df : 40
Missing curated_df   : 6

Missing combos:
  - ai_usage_technology_literacy × strategic_injected_tag
  - climate_sustainability × state_cluster
  - hunger_food_security_food_systems × urbanity
  - lgbtq × state_cluster
  - rural_school_needs × strategic_injected_tag
  - safety_justice × strategic_injected_tag

Paste-ready YAML for missing combos:
────────────────────────────────────
run_plan:
  ai_usage_technology_literacy:
    first_split: strategic_injected_tag
  climate_sustainability:
    second_splits:
      - state_cluster
  hunger_food_security_food_systems:
    second_splits:
      - urbanity
  lgbtq:
    second_splits:
      - state_cluster
  rural_school_needs:
    first_split: strategic_injected_tag
  safety_justice:
    first_split: strategic_injected_tag


/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_38652/3244658734.py:172: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  status["has_curated_df"] = status["has_curated_df"].fillna(False)


In [1]:
# UNION INSIGHTS AFTER SWEEP TO CREATE build_insight_ranking_input.py

"""
Find every curated_df.csv under OUTPUTS/runs/strategic_area, combine them,
and write one slim ranking-input JSONL for LLM scoring/ranking.

This is intended for the post-sweep ranking step: one record per accepted
insight, including appendix-tagged insights, with only the fields the ranking
LLM needs.

Usage
-----
    python build_insight_ranking_input.py

Outputs
-------
    OUTPUTS/sweep_review/insight_ranking_input.jsonl
    OUTPUTS/sweep_review/insight_ranking_input.csv
    OUTPUTS/sweep_review/insight_ranking_input_manifest.json
"""

from __future__ import annotations

import json
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd


# ── CONFIG ────────────────────────────────────────────────────────────────────
ROOT_DIR = Path(
    r"/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area"
)

DEST_DIR = Path(
    r"/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review"
)

FILENAME = "curated_df.csv"

OUT_JSONL = "insight_ranking_input.jsonl"
OUT_CSV = "insight_ranking_input.csv"
OUT_MANIFEST = "insight_ranking_input_manifest.json"

DEBUG = False
# ─────────────────────────────────────────────────────────────────────────────


# Fields to pass to the LLM.
#
# Excluded per your instruction:
# run_id, strategic_area_label, split_id, resolved_groupby_field, insight_id,
# section, report_section, verified_topic_count, claimed_topic_count,
# verification_ratio, source_topics_verified.
RANKING_FIELDS = [
    "global_insight_id",
    "strategic_area_id",
    "category_bucket",
    "title",
    "finding",
    "evidence_basis",
    "scope_or_caveat",
    "why_it_matters",
    "supporting_project_count",
    "mean_topic_share_all_verified_topics",
]


def slugify(value: object, max_len: int = 80) -> str:
    """Convert an arbitrary value into a compact ID-safe slug."""
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return (s[:max_len] or "missing")


def find_curated_files(root_dir: Path) -> list[Path]:
    """Return all curated_df.csv files under root_dir."""
    matches: list[Path] = []

    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower() == FILENAME.lower():
                full_path = Path(dirpath) / fname
                matches.append(full_path)

                if DEBUG:
                    modified = datetime.fromtimestamp(full_path.stat().st_mtime)
                    print(f"FOUND: {full_path}")
                    print(f"       modified = {modified:%Y-%m-%d %I:%M %p}\n")

    return sorted(matches)


def read_one_curated_file(path: Path, root_dir: Path) -> pd.DataFrame:
    """Read one curated_df.csv and add source-file metadata."""
    df = pd.read_csv(path)
    rel_path = path.relative_to(root_dir)

    df["_source_file"] = str(path)
    df["_source_rel_path"] = str(rel_path)
    df["_source_modified_at"] = datetime.fromtimestamp(
        path.stat().st_mtime
    ).isoformat(timespec="seconds")

    return df


def first_present(row: pd.Series, candidates: list[str], default: object = "") -> object:
    """Return the first non-null/non-empty value among candidate columns."""
    for col in candidates:
        if col in row.index:
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                return val
    return default


def add_global_insight_id(df: pd.DataFrame) -> pd.DataFrame:
    """Add a stable global_insight_id for downstream ranking traceability.

    Uses existing run/context fields when available, but does not pass those
    fields through to the LLM ranking input.
    """
    out = df.copy()

    def _make_id(row: pd.Series, idx: int) -> str:
        strategic_area_id = first_present(row, ["strategic_area_id"], "area")
        split_id = first_present(row, ["split_id"], "split")
        insight_id = first_present(row, ["insight_id"], "")
        category_bucket = first_present(row, ["category_bucket"], "")
        title = first_present(row, ["title"], "")

        if insight_id:
            insight_part = slugify(insight_id)
        else:
            insight_part = f"row_{idx:06d}"

        return "__".join(
            [
                slugify(strategic_area_id),
                slugify(split_id),
                slugify(category_bucket, max_len=50),
                insight_part,
                slugify(title, max_len=70),
            ]
        )

    out["global_insight_id"] = [
        _make_id(row, idx) for idx, (_, row) in enumerate(out.iterrows(), start=1)
    ]

    # If duplicate IDs arise, suffix them deterministically.
    dup_count = out.groupby("global_insight_id").cumcount()
    out["global_insight_id"] = out["global_insight_id"].where(
        dup_count.eq(0),
        out["global_insight_id"] + "__dup_" + dup_count.astype(str),
    )

    return out


def coerce_numeric_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce ranking metric fields to numeric where present."""
    out = df.copy()

    numeric_cols = [
        "supporting_project_count",
        "mean_topic_share_all_verified_topics",
    ]

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def build_ranking_input(combined_df: pd.DataFrame) -> pd.DataFrame:
    """Return the slim LLM ranking input dataframe."""
    work = combined_df.copy()
    work = add_global_insight_id(work)
    work = coerce_numeric_fields(work)

    # Ensure expected LLM fields exist even if one older curated_df lacks one.
    for col in RANKING_FIELDS:
        if col not in work.columns:
            work[col] = pd.NA

    ranking_df = work[RANKING_FIELDS].copy()

    # Basic cleanup so JSONL stays compact and predictable.
    text_cols = [
        "global_insight_id",
        "strategic_area_id",
        "category_bucket",
        "title",
        "finding",
        "evidence_basis",
        "scope_or_caveat",
        "why_it_matters",
    ]
    for col in text_cols:
        ranking_df[col] = (
            ranking_df[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    # Drop rows with no substantive insight text.
    substantive = (
        ranking_df["title"].ne("")
        | ranking_df["finding"].ne("")
        | ranking_df["evidence_basis"].ne("")
    )
    ranking_df = ranking_df[substantive].reset_index(drop=True)

    return ranking_df


def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    """Write dataframe records as UTF-8 JSONL."""
    with open(path, "w", encoding="utf-8") as f:
        for rec in df.to_dict(orient="records"):
            clean = {
                k: (None if pd.isna(v) else v)
                for k, v in rec.items()
            }
            f.write(json.dumps(clean, ensure_ascii=False) + "\n")


def main() -> None:
    print(f"Looking for {FILENAME} under:\n  {ROOT_DIR}\n")

    if not ROOT_DIR.exists():
        raise FileNotFoundError(f"ROOT_DIR does not exist: {ROOT_DIR}")

    DEST_DIR.mkdir(parents=True, exist_ok=True)

    matches = find_curated_files(ROOT_DIR)

    if not matches:
        print("No curated_df.csv files found.")
        return

    print(f"Found {len(matches):,} curated_df.csv file(s).")

    frames = []
    read_errors = []

    for path in matches:
        try:
            frames.append(read_one_curated_file(path, ROOT_DIR))
        except Exception as e:
            read_errors.append({
                "path": str(path),
                "error": f"{type(e).__name__}: {e}",
            })
            print(f"  ! Failed to read: {path}")
            print(f"    {type(e).__name__}: {e}")

    if not frames:
        raise RuntimeError("No curated_df.csv files could be read successfully.")

    combined_df = pd.concat(frames, ignore_index=True)
    ranking_df = build_ranking_input(combined_df)

    out_jsonl = DEST_DIR / OUT_JSONL
    out_csv = DEST_DIR / OUT_CSV
    out_manifest = DEST_DIR / OUT_MANIFEST

    write_jsonl(ranking_df, out_jsonl)
    ranking_df.to_csv(out_csv, index=False)

    manifest = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "root_dir": str(ROOT_DIR),
        "dest_dir": str(DEST_DIR),
        "filename_searched": FILENAME,
        "files_found": len(matches),
        "files_read_successfully": len(frames),
        "files_read_failed": len(read_errors),
        "input_rows_combined": int(len(combined_df)),
        "ranking_rows_written": int(len(ranking_df)),
        "output_jsonl": str(out_jsonl),
        "output_csv": str(out_csv),
        "read_errors": read_errors,
        "source_files": [str(p) for p in matches],
        "fields_written": RANKING_FIELDS,
        "fields_excluded_by_design": [
            "run_id",
            "strategic_area_label",
            "split_id",
            "resolved_groupby_field",
            "insight_id",
            "section",
            "report_section",
            "verified_topic_count",
            "claimed_topic_count",
            "verification_ratio",
            "source_topics_verified",
        ],
    }

    out_manifest.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("\nDone.")
    print(f"  Files found           : {len(matches):,}")
    print(f"  Files read            : {len(frames):,}")
    print(f"  Combined input rows   : {len(combined_df):,}")
    print(f"  Ranking rows written  : {len(ranking_df):,}")
    print(f"  JSONL                 : {out_jsonl}")
    print(f"  CSV                   : {out_csv}")
    print(f"  Manifest              : {out_manifest}")


if __name__ == "__main__":
    main()

Looking for curated_df.csv under:
  /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area

Found 46 curated_df.csv file(s).

Done.
  Files found           : 46
  Files read            : 46
  Combined input rows   : 703
  Ranking rows written  : 703
  JSONL                 : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review/insight_ranking_input.jsonl
  CSV                   : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review/insight_ranking_input.csv
  Manifest              : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review/insight_ranking_input_manifest.json


In [ ]:
# SCORE INSIGHT RANKING INPUT VIA OPENAI RESPONSES API
#
# Replacement cell.
#
# Keeps your prompts unchanged, but replaces the API plumbing with:
#   - OpenAI Responses API
#   - strict JSON schema output
#   - flex -> default service-tier fallback
#   - per-batch payload validation
#   - timestamped JSON + CSV outputs
#
# Required:
#   pip install -U openai
#   export OPENAI_API_KEY=...

from __future__ import annotations

import csv
import json
import math
import os
import random
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import httpx
from openai import OpenAI


# ── CONFIG ────────────────────────────────────────────────────────────────────
SWEEP_DIR = Path(
    r"/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review"
)
INPUT_JSONL = SWEEP_DIR / "insight_ranking_input.jsonl"

MODEL = "gpt-5.4"            # or "gpt-5.4-mini" for cheaper rubric scoring
REASONING_EFFORT = "medium"       # "low" is probably fine after testing
SERVICE_TIER = "flex"
FALLBACK_SERVICE_TIER = "default"

BATCH_SIZE = 50
BATCH_LIMIT: int | None = None        # set to None for full run
SHUFFLE_SEED: int | None = 42      # set to None to preserve source order

MAX_RETRIES = 20
RETRY_BACKOFF_S = 5
TIMEOUT_S = 900

# Leave as None because you said you do not care about max tokens.
MAX_OUTPUT_TOKENS: int | None = None

STAMP = datetime.now().strftime("%y%m%d_%H%M")
OUT_JSON = SWEEP_DIR / f"insight_scores_{STAMP}.json"
OUT_CSV = SWEEP_DIR / f"insight_scores_{STAMP}.csv"
OUT_PARTIAL_JSON = SWEEP_DIR / f"insight_scores_{STAMP}.partial.json"
# ─────────────────────────────────────────────────────────────────────────────


# ── PROMPTS: PASTE YOUR EXACT EXISTING PROMPTS HERE, UNCHANGED ────────────────
SYSTEM_PROMPT = """\
You are scoring DonorsChoose-generated classroom insight records for
business usefulness. Each insight is a structured finding distilled from
clusters of teacher funding requests in DonorsChoose's "Classroom Compass"
report.

Your task: given a batch of insight records, return one score object per
input record. Each output object contains global_insight_id and five
integer scores in [0, 5]: actionability, evidence_strength, novelty,
scope_leverage, clarity.

OUTPUT FORMAT — REQUIRED

Return one JSON object with this exact shape. No prose, no markdown
fences, no extra fields.

{
  "scores": [
    {
      "global_insight_id": "<exact id from the corresponding input record>",
      "actionability":     <integer 0..5>,
      "evidence_strength": <integer 0..5>,
      "novelty":           <integer 0..5>,
      "scope_leverage":    <integer 0..5>,
      "clarity":           <integer 0..5>
    }
  ]
}

Rules for the output:
- "scores" must have exactly one object per input record.
- The order of objects in "scores" must match the order of the input records.
- "global_insight_id" must be copied verbatim from the input — never modified,
  abbreviated, or generated. Every input ID must appear exactly once.
- All five score fields must be integers in [0, 5]. Not strings, not floats,
  not null.
- Do not add any keys beyond the six listed. Do not add a top-level field
  besides "scores".

CORPUS CONTEXT
- supporting_project_count in this corpus ranges from ~60 to ~13,000
  (median ~1,150). Treat <200 as low volume, 200-800 moderate,
  800-3,000 strong, >3,000 very strong.
- mean_topic_share_all_verified_topics ranges from ~0.40 to ~0.86
  (median ~0.45). Treat <0.45 diffuse, 0.45-0.55 moderately concentrated,
  >0.55 concentrated.
- category_bucket may be blank, null, missing, or empty. When it is,
  treat the record as a cross-category insight for its strategic_area_id.
  Cross-category insights generally receive higher scope_leverage when
  the underlying evidence supports a broad pattern.

RUBRIC

Use the full 0-5 range. Do not cluster at 3-4. If you are tempted to give
every insight a 4, you are not discriminating; re-anchor against the
rubric.

actionability — Can DonorsChoose plausibly do something specific with this?
  5  Clearly suggests a practical program, partnership, campaign,
     product, vendor relationship, merchandising decision, fulfillment
     change, reporting structure, or research direction. Decision target
     is unambiguous (e.g., "track these separately in reporting,"
     "fund X alongside Y," "pair grants with training").
  4  Reasonably actionable; implementation path needs some interpretation.
  3  Somewhat actionable but broad, indirect, or exploratory.
  2  Weakly actionable; mostly descriptive.
  1  Restates the finding without proposing action.
  0  No actionable content.

evidence_strength — How well does the evidence justify the claim?
  Inputs: supporting_project_count, mean_topic_share, specificity of
  evidence_basis (named tools, programs, routines vs vague references),
  and coherence between evidence and claim. Penalize when the claim is
  broader than the evidence (e.g., a finding about "classrooms" supported
  only by evidence about one grade band).
  5  Very strong on all axes: volume >=3,000, share >=0.55, and
     evidence_basis names specific tools or routines, coherent with claim.
  4  Strong on two of {volume >=1,000, named specifics, share >=0.50}.
  3  Moderate evidence on at least one axis; mid-volume (500-1,500) with
     reasonable specifics.
  2  Some evidence but mostly generic; count 200-500, or claim somewhat
     broader than evidence.
  1  Thin evidence — count <200, or evidence_basis is vague.
  0  Evidence_basis does not actually support the finding.

  Do not score evidence_strength mechanically from volume alone. A
  lower-volume insight can still score 4 if the evidence is highly
  specific and coherent; reserve 5 for unusually strong support across
  volume, concentration, and specificity.

novelty — Non-obvious, or a generic restatement?
  5  Specific, surprising, or strategically distinctive. Surfaces a
     genuine distinction or counterintuitive structural pattern.
  4  Names a real distinction or splits a category usefully ("X splits
     into two different functions"; "Y looks like Z but actually serves W").
  3  A reasonable observation a thoughtful reader could arrive at.
  2  True but obvious — restates what the category implies, or generic
     ("teachers need supplies", "students need engagement",
     "schools need resources").
  1  Extremely obvious.
  0  Tautological or empty.

scope_leverage — Applies broadly or to a repeatable pattern?
  5  Cross-category insight (blank/null/empty category_bucket) with volume
     >=2,000 AND a coherent pattern that clearly generalizes across
     categories.
  4  Cross-category with a credible but less distinctive pattern, OR a
     very high-volume in-category pattern (>=1,500); meaningful segment
     with likely repeatability.
  3  In-category pattern recurring across many projects (500-1,500).
  2  Narrower pattern, moderate volume (200-500); localized.
  1  Niche, narrow segment, or anecdotal (<200).
  0  One-off observation; no discernible scope.

clarity — Board/funder/operator-ready?
  5  Title and why_it_matters are concrete and specific; each field adds
     new information (no repetition across finding/evidence/why);
     scope_or_caveat genuinely narrows the claim.
  4  Clear and reasonably structured; minor wording or redundancy issues.
  3  Understandable but wordy, vague, or finding and why_it_matters say
     similar things, or scope_or_caveat is boilerplate.
  2  Hard to parse or jargon-heavy.
  1  Confusing or contradictory.
  0  Incoherent.

OPERATIONAL RULES
- Use the full record, not just the title or finding.
- Score each insight independently. Do not let one score influence others.
- Output order MUST match input order. Process every record. Skip none.
- Do not invent evidence beyond the provided record.
- Penalize generic claims ("teachers need supplies," "students need
  engagement," "schools need resources") unless the record adds a
  specific, non-obvious, actionable pattern.
- Penalize evidence_strength when the claim is broader than the evidence.
- Penalize clarity when wording is not ready for a board, funder, or
  operator audience.
- If a field is missing, do not invent it. Use the remaining fields
  where possible. Score 0 only when the missing field makes that
  dimension impossible to assess.
"""


USER_TEMPLATE = """\
Score the following DonorsChoose insight records.

Input format:
- JSONL, one insight record per line.
- Each record includes a global_insight_id.
- category_bucket may be blank/null/missing; when it is, treat the record
  as a cross-category insight for its strategic_area_id.

This is batch {{BATCH_NUMBER}} of {{TOTAL_BATCHES}}. Score every record in
this batch only.

Return only the JSON object described in the OUTPUT FORMAT section.

INSIGHT_RECORDS_JSONL:
<<<
{{INSIGHTS_BATCH_JSONL}}
>>>
"""
# ─────────────────────────────────────────────────────────────────────────────


RESPONSE_SCHEMA = {
    "type": "json_schema",
    "name": "insight_score_batch",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "required": ["scores"],
        "properties": {
            "scores": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": [
                        "global_insight_id",
                        "actionability",
                        "evidence_strength",
                        "novelty",
                        "scope_leverage",
                        "clarity",
                    ],
                    "properties": {
                        "global_insight_id": {"type": "string"},
                        "actionability": {
                            "type": "integer",
                            "minimum": 0,
                            "maximum": 5,
                        },
                        "evidence_strength": {
                            "type": "integer",
                            "minimum": 0,
                            "maximum": 5,
                        },
                        "novelty": {
                            "type": "integer",
                            "minimum": 0,
                            "maximum": 5,
                        },
                        "scope_leverage": {
                            "type": "integer",
                            "minimum": 0,
                            "maximum": 5,
                        },
                        "clarity": {
                            "type": "integer",
                            "minimum": 0,
                            "maximum": 5,
                        },
                    },
                },
            },
        },
    },
}

REQUIRED_KEYS = [
    "global_insight_id",
    "actionability",
    "evidence_strength",
    "novelty",
    "scope_leverage",
    "clarity",
]
SCORE_KEYS = REQUIRED_KEYS[1:]


def make_openai_client() -> OpenAI:
    """Create an OpenAI client.

    Uses verify=False to match the DonorsChoose proxy pattern from the
    pipeline utilities. If you do not need that locally, change to OpenAI().
    """
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")

    return OpenAI(
        api_key=api_key,
        timeout=TIMEOUT_S,
        http_client=httpx.Client(verify=False, timeout=TIMEOUT_S),
    )


def build_user_message(batch: list[dict[str, Any]], batch_idx: int, total_batches: int) -> str:
    batch_jsonl = "\n".join(
        json.dumps(ins, ensure_ascii=False, default=str)
        for ins in batch
    )

    return (
        USER_TEMPLATE
        .replace("{{BATCH_NUMBER}}", str(batch_idx + 1))
        .replace("{{TOTAL_BATCHES}}", str(total_batches))
        .replace("{{INSIGHTS_BATCH_JSONL}}", batch_jsonl)
    )


def parse_response_payload(resp: Any) -> dict[str, Any]:
    """Extract JSON payload from a Responses API response."""
    status = getattr(resp, "status", None)
    if status and status != "completed":
        incomplete = getattr(resp, "incomplete_details", None)
        err = getattr(resp, "error", None)
        raise RuntimeError(f"Response status={status}; incomplete={incomplete}; error={err}")

    text = getattr(resp, "output_text", None)
    if not text:
        raise RuntimeError("Responses API returned empty output_text.")

    return json.loads(text)


def validate_scores(payload: dict[str, Any], expected_ids: list[str]) -> list[dict[str, Any]]:
    """Strict local validation even though the API is also schema-constrained."""
    if not isinstance(payload, dict):
        raise ValueError(f"top-level response must be object, got {type(payload).__name__}")

    scores = payload.get("scores")
    if not isinstance(scores, list):
        raise ValueError("'scores' must be a list")

    if len(scores) != len(expected_ids):
        raise ValueError(f"expected {len(expected_ids)} scores, got {len(scores)}")

    for i, score in enumerate(scores):
        if not isinstance(score, dict):
            raise ValueError(f"scores[{i}] must be object, got {type(score).__name__}")

        extra = sorted(set(score) - set(REQUIRED_KEYS))
        missing = [k for k in REQUIRED_KEYS if k not in score]
        if extra:
            raise ValueError(f"scores[{i}] has extra keys: {extra}")
        if missing:
            raise ValueError(f"scores[{i}] missing keys: {missing}")

        if score["global_insight_id"] != expected_ids[i]:
            raise ValueError(
                f"scores[{i}] id/order mismatch\n"
                f"  expected: {expected_ids[i]}\n"
                f"  got:      {score['global_insight_id']}"
            )

        for key in SCORE_KEYS:
            value = score[key]
            if isinstance(value, bool) or not isinstance(value, int):
                raise ValueError(
                    f"scores[{i}].{key} must be int 0..5, got {type(value).__name__}: {value!r}"
                )
            if value < 0 or value > 5:
                raise ValueError(f"scores[{i}].{key} out of range: {value}")

    return scores


def create_response_once(
    client: OpenAI,
    *,
    system_prompt: str,
    user_message: str,
    service_tier: str,
) -> Any:
    kwargs = {
        "model": MODEL,
        "instructions": system_prompt,
        "input": user_message,
        "reasoning": {"effort": REASONING_EFFORT},
        "service_tier": service_tier,
        "text": {"format": RESPONSE_SCHEMA},
        "store": False,
    }

    if MAX_OUTPUT_TOKENS is not None:
        kwargs["max_output_tokens"] = MAX_OUTPUT_TOKENS

    return client.responses.create(**kwargs)


def score_one_batch(
    client: OpenAI,
    batch: list[dict[str, Any]],
    batch_idx: int,
    total_batches: int,
) -> list[dict[str, Any]]:
    user_message = build_user_message(batch, batch_idx, total_batches)
    expected_ids = [str(ins["global_insight_id"]) for ins in batch]

    last_err: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 1):
        for tier in (SERVICE_TIER, FALLBACK_SERVICE_TIER):
            try:
                resp = create_response_once(
                    client,
                    system_prompt=SYSTEM_PROMPT,
                    user_message=user_message,
                    service_tier=tier,
                )
                payload = parse_response_payload(resp)
                return validate_scores(payload, expected_ids)

            except Exception as e:
                last_err = e
                print(
                    f"    batch {batch_idx + 1}, attempt {attempt}/{MAX_RETRIES}, "
                    f"tier={tier}: {type(e).__name__}: {e}"
                )

                # If flex fails, immediately try default in the same attempt.
                # If default fails too, sleep and start the next attempt.
                if tier == SERVICE_TIER and FALLBACK_SERVICE_TIER != SERVICE_TIER:
                    continue

                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_BACKOFF_S)
                break

    raise RuntimeError(
        f"Batch {batch_idx + 1} failed after {MAX_RETRIES} attempts."
    ) from last_err


def write_outputs(scores: list[dict[str, Any]], json_path: Path, csv_path: Path) -> None:
    json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(scores, f, indent=2, ensure_ascii=False)

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=REQUIRED_KEYS)
        writer.writeheader()
        writer.writerows(scores)


def main() -> None:
    if "PASTE YOUR EXACT EXISTING SYSTEM_PROMPT HERE" in SYSTEM_PROMPT:
        raise RuntimeError("Paste the exact existing SYSTEM_PROMPT into this cell before running.")
    if "PASTE YOUR EXACT EXISTING USER_TEMPLATE HERE" in USER_TEMPLATE:
        raise RuntimeError("Paste the exact existing USER_TEMPLATE into this cell before running.")
    if not INPUT_JSONL.exists():
        raise FileNotFoundError(f"INPUT_JSONL does not exist: {INPUT_JSONL}")

    with open(INPUT_JSONL, encoding="utf-8") as f:
        insights = [json.loads(line) for line in f if line.strip()]

    if not insights:
        raise RuntimeError(f"No insight records found in {INPUT_JSONL}")

    print(f"Loaded {len(insights):,} insights from {INPUT_JSONL}")

    if SHUFFLE_SEED is not None:
        random.Random(SHUFFLE_SEED).shuffle(insights)
        print(f"Shuffled with seed={SHUFFLE_SEED}")

    total_batches = math.ceil(len(insights) / BATCH_SIZE)
    batches_to_run = total_batches if BATCH_LIMIT is None else min(total_batches, BATCH_LIMIT)

    if batches_to_run < total_batches:
        print(f"DRY RUN: scoring {batches_to_run} of {total_batches} batches")
    else:
        print(f"Scoring all {total_batches} batches")

    print(
        f"Model={MODEL}; reasoning.effort={REASONING_EFFORT}; "
        f"tier={SERVICE_TIER}->{FALLBACK_SERVICE_TIER}; batch_size={BATCH_SIZE}"
    )

    client = make_openai_client()
    all_scores: list[dict[str, Any]] = []

    for batch_idx in range(batches_to_run):
        start = batch_idx * BATCH_SIZE
        batch = insights[start:start + BATCH_SIZE]

        batch_scores = score_one_batch(
            client=client,
            batch=batch,
            batch_idx=batch_idx,
            total_batches=total_batches,
        )

        all_scores.extend(batch_scores)

        # Checkpoint after every successful batch.
        with open(OUT_PARTIAL_JSON, "w", encoding="utf-8") as f:
            json.dump(all_scores, f, indent=2, ensure_ascii=False)

        print(f"  batch {batch_idx + 1}/{total_batches}: {len(batch_scores)} scored")

    # Whole-run integrity checks when not dry-running.
    if BATCH_LIMIT is None:
        input_ids = [str(ins["global_insight_id"]) for ins in insights]
        output_ids = [str(sc["global_insight_id"]) for sc in all_scores]

        assert len(output_ids) == len(input_ids), (
            f"length mismatch: input={len(input_ids)}, output={len(output_ids)}"
        )
        assert len(output_ids) == len(set(output_ids)), "duplicate output ids"

        missing = set(input_ids) - set(output_ids)
        extra = set(output_ids) - set(input_ids)
        assert not missing and not extra, (
            f"id set mismatch: missing={len(missing)} extra={len(extra)}\n"
            f"first missing: {sorted(missing)[:5]}\n"
            f"first extra:   {sorted(extra)[:5]}"
        )

    write_outputs(all_scores, OUT_JSON, OUT_CSV)

    print("\nDone.")
    print(f"  Insights scored : {len(all_scores):,}")
    print(f"  JSON output     : {OUT_JSON}")
    print(f"  CSV output      : {OUT_CSV}")
    print(f"  Partial backup  : {OUT_PARTIAL_JSON}")


if __name__ == "__main__":
    main()

Loaded 703 insights from /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/sweep_review/insight_ranking_input.jsonl
Shuffled with seed=42
Scoring all 15 batches
Model=gpt-5.4; reasoning.effort=medium; tier=flex->default; batch_size=50
